# Day 2: Chunking Implementation

Goal: take the 5,183 SciFact abstracts saved in `corpus.json` and produce six parallel chunked versions of the corpus, one per strategy:

1. **Document-level** — one chunk per abstract, with no splitting.
2. **Sentence-level** — one chunk per abstract sentence, with the document title prepended for retrieval signal.
3. **Fixed-token 128** — split into fixed 128-token windows with 20-token overlap, without trying to preserve sentence or paragraph boundaries.
4. **Recursive 128** — split into approximately 128-token chunks with 20-token overlap while trying to preserve natural boundaries such as sentences and words.
5. **Recursive 256** — split into approximately 256-token chunks with 40-token overlap, giving a less aggressive recursive baseline.
6. **Semantic chunking** — split based on semantic shifts between sentence groups using embedding similarity.

Each output chunk carries at least `{chunk_id, doc_id, text}` so we can evaluate retrieval against SciFact gold evidence document IDs later. Some strategies also include extra metadata such as `sentence_id`, `chunk_index`, and `chunk_type`.

Outputs, one JSON file per strategy:

```text
chunks/chunks_document.json
chunks/chunks_sentence.json
chunks/chunks_fixed_128.json
chunks/chunks_recursive_128.json
chunks/chunks_recursive_256.json
chunks/chunks_semantic.json

## 0. Install dependencies

Run once. The semantic chunker needs an embedding model — we use `bge-small-en-v1.5` here so it matches what we'll use on Day 3 for indexing.

In [1]:
%pip install -q langchain-text-splitters langchain-experimental langchain-huggingface sentence-transformers tiktoken pandas tqdm


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

from langchain_text_splitters import RecursiveCharacterTextSplitter, TokenTextSplitter

## 1. Load the cleaned corpus from Day 1

In [4]:
CORPUS_PATH = Path("corpus.json")
CHUNK_DIR = Path("chunks")
CHUNK_DIR.mkdir(exist_ok=True)

with open(CORPUS_PATH, "r") as f:
    corpus = json.load(f)

print(f"Loaded {len(corpus)} documents")
print(f"Sample doc keys: {list(corpus[0].keys())}")
print(f"Sample title: {corpus[0]['title']}")

Loaded 5183 documents
Sample doc keys: ['doc_id', 'title', 'abstract_sentences', 'abstract_text', 'structured']
Sample title: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.


## 2. Helper: save chunks and report stats

All four strategies will produce a list of `{chunk_id, doc_id, text}` dicts. This helper writes them out and prints sanity-check stats.

In [5]:
def get_doc_id(doc):
    return str(doc["doc_id"])


def get_title(doc):
    return doc.get("title", "").strip()


def get_abstract_text(doc):
    """
    Uses abstract_text if available.
    Otherwise reconstructs it from abstract_sentences.
    """
    if "abstract_text" in doc and doc["abstract_text"]:
        return doc["abstract_text"].strip()

    if "abstract_sentences" in doc and doc["abstract_sentences"]:
        return " ".join(doc["abstract_sentences"]).strip()

    return ""


def get_abstract_sentences(doc):
    """
    Uses abstract_sentences if available.
    Otherwise falls back to a rough sentence split.
    """
    if "abstract_sentences" in doc and doc["abstract_sentences"]:
        return [s.strip() for s in doc["abstract_sentences"] if s.strip()]

    abstract_text = get_abstract_text(doc)
    return [s.strip() for s in abstract_text.split(". ") if s.strip()]


def make_full_text(doc):
    """
    Prepends title to abstract.
    This is useful because title carries strong retrieval signal.
    """
    title = get_title(doc)
    abstract = get_abstract_text(doc)

    if title and abstract:
        return f"{title}. {abstract}".strip()

    if title:
        return title

    return abstract


def save_and_report(chunks, name, path):
    path = Path(path)

    with open(path, "w") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    lengths = np.array([len(c["text"].split()) for c in chunks])
    docs_covered = len({str(c["doc_id"]) for c in chunks})

    print(f"=== {name} ===")
    print(f"  Total chunks:    {len(chunks)}")
    print(f"  Unique docs:     {docs_covered} / {len(corpus)}")
    print(f"  Chunks/doc:      {len(chunks) / len(corpus):.2f}")
    print(f"  Avg words/chunk: {lengths.mean():.1f}")
    print(f"  Median:          {np.median(lengths):.0f}")
    print(f"  Min / Max:       {lengths.min()} / {lengths.max()}")
    print(f"  Saved to:        {path}")
    print()

## 3. Strategy 1 — Document-level (no splitting)

Each abstract becomes a single chunk. Day 1 stats showed 99.4% of abstracts are under 512 words, so most will fit comfortably in a 512-token embedding context. This is the natural baseline.

In [6]:
document_chunks = []

for doc in corpus:
    doc_id = get_doc_id(doc)
    text = make_full_text(doc)

    if not text:
        continue

    document_chunks.append({
        "chunk_id": f"{doc_id}_doc_0",
        "doc_id": doc_id,
        "chunk_index": 0,
        "chunk_type": "document",
        "text": text,
    })

save_and_report(
    document_chunks,
    "Document-level",
    CHUNK_DIR / "chunks_document.json"
)

=== Document-level ===
  Total chunks:    5183
  Unique docs:     5183 / 5183
  Chunks/doc:      1.00
  Avg words/chunk: 214.6
  Median:          204
  Min / Max:       33 / 1541
  Saved to:        chunks/chunks_document.json



## 4. Strategy 2 — Sentence-level chunking

Each abstract is split into sentences, and each sentence becomes its own chunk.  
This gives smaller, more focused chunks than document-level chunking, which may help retrieval when queries match specific claims or details inside an abstract.


In [8]:
sentence_chunks = []

for doc in corpus:
    doc_id = get_doc_id(doc)
    title = get_title(doc)
    sentences = get_abstract_sentences(doc)

    for sent_id, sentence in enumerate(sentences):
        if not sentence.strip():
            continue

        if title:
            text = f"{title}. {sentence}".strip()
        else:
            text = sentence.strip()

        sentence_chunks.append({
            "chunk_id": f"{doc_id}_sent_{sent_id}",
            "doc_id": doc_id,
            "sentence_id": sent_id,
            "chunk_index": sent_id,
            "chunk_type": "sentence",
            "text": text,
        })

save_and_report(
    sentence_chunks,
    "Sentence-level",
    CHUNK_DIR / "chunks_sentence.json"
)

=== Sentence-level ===
  Total chunks:    45952
  Unique docs:     5183 / 5183
  Chunks/doc:      8.87
  Avg words/chunk: 35.9
  Median:          35
  Min / Max:       3 / 221
  Saved to:        chunks/chunks_sentence.json



## 4. Strategy 3 — Fixed-token chunking

Split each document into fixed-size token windows, without trying to preserve sentence or paragraph boundaries. This is the deliberately naive baseline.

Unlike recursive chunking, this method does not search for natural split points such as sentence endings. It simply creates chunks of approximately 128 tokens with a 20-token overlap. The overlap helps preserve local context when important evidence falls near a chunk boundary.

This strategy is useful as a control: it tells us how much performance we get from simple length-based token windows compared with structure-aware recursive chunking and sentence-level chunking.

In [9]:
fixed_token_splitter = TokenTextSplitter(
    chunk_size=128,
    chunk_overlap=20,
)

fixed_token_chunks = []

for doc in corpus:
    doc_id = get_doc_id(doc)
    text = make_full_text(doc)

    if not text:
        continue

    pieces = fixed_token_splitter.split_text(text)

    for i, piece in enumerate(pieces):
        fixed_token_chunks.append({
            "chunk_id": f"{doc_id}_fixed128_{i}",
            "doc_id": doc_id,
            "chunk_index": i,
            "chunk_type": "fixed_token_128",
            "text": piece.strip(),
        })

save_and_report(
    fixed_token_chunks,
    "Fixed-token 128",
    CHUNK_DIR / "chunks_fixed_128.json"
)

=== Fixed-token 128 ===
  Total chunks:    17400
  Unique docs:     5183 / 5183
  Chunks/doc:      3.36
  Avg words/chunk: 73.3
  Median:          78
  Min / Max:       5 / 125
  Saved to:        chunks/chunks_fixed_128.json



## 5. Strategy 4 — Recursive 128-token chunking

LangChain's `RecursiveCharacterTextSplitter` tries to split text at natural boundaries: paragraph breaks, line breaks, sentence endings, words, and finally characters if needed.

Although the class name says “Character,” we use `from_tiktoken_encoder`, so chunk size is measured in tokens. Since most SciFact abstracts are much shorter than 512 tokens, we use smaller budgets: 128 tokens with 20-token overlap and 256 tokens with 40-token overlap. This makes recursive chunking meaningfully different from document-level retrieval while still preserving natural text boundaries better than fixed-token splitting.

In [10]:
recursive_128_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=128,
    chunk_overlap=20,
    separators=["\n\n", "\n", ". ", " ", ""],
)

recursive_128_chunks = []

for doc in corpus:
    doc_id = get_doc_id(doc)
    text = make_full_text(doc)

    if not text:
        continue

    pieces = recursive_128_splitter.split_text(text)

    for i, piece in enumerate(pieces):
        recursive_128_chunks.append({
            "chunk_id": f"{doc_id}_rec128_{i}",
            "doc_id": doc_id,
            "chunk_index": i,
            "chunk_type": "recursive_128",
            "text": piece.strip(),
        })

save_and_report(
    recursive_128_chunks,
    "Recursive 128 tokens",
    CHUNK_DIR / "chunks_recursive_128.json"
)

=== Recursive 128 tokens ===
  Total chunks:    18852
  Unique docs:     5183 / 5183
  Chunks/doc:      3.64
  Avg words/chunk: 60.6
  Median:          63
  Min / Max:       1 / 117
  Saved to:        chunks/chunks_recursive_128.json



## 6. Strategy 5 — Recursive token-aware splitting, 256-token chunks

This is the medium-sized recursive chunking setting. Like the 128-token version, it uses LangChain's `RecursiveCharacterTextSplitter` with `from_tiktoken_encoder`, so the chunk budget is measured in tokens rather than raw characters.

The splitter tries to preserve natural boundaries in this order: paragraph breaks, line breaks, sentence endings, words, and finally characters if needed.

We use 256-token chunks with a 40-token overlap. This setting is less aggressive than 128-token chunking, so it keeps more abstract-level context while still splitting longer SciFact abstracts when needed. It is useful as a compromise between full-document retrieval and smaller evidence-focused chunks.

In [11]:
recursive_256_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=256,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""],
)

recursive_256_chunks = []

for doc in corpus:
    doc_id = get_doc_id(doc)
    text = make_full_text(doc)

    if not text:
        continue

    pieces = recursive_256_splitter.split_text(text)

    for i, piece in enumerate(pieces):
        recursive_256_chunks.append({
            "chunk_id": f"{doc_id}_rec256_{i}",
            "doc_id": doc_id,
            "chunk_index": i,
            "chunk_type": "recursive_256",
            "text": piece.strip(),
        })

save_and_report(
    recursive_256_chunks,
    "Recursive 256 tokens",
    CHUNK_DIR / "chunks_recursive_256.json"
)

=== Recursive 256 tokens ===
  Total chunks:    10011
  Unique docs:     5183 / 5183
  Chunks/doc:      1.93
  Avg words/chunk: 115.4
  Median:          125
  Min / Max:       1 / 223
  Saved to:        chunks/chunks_recursive_256.json



## 7. Strategy 6 — Semantic chunking

Semantic chunking splits text based on changes in meaning rather than fixed length or sentence count. We use LangChain's `SemanticChunker` with `BAAI/bge-small-en-v1.5` embeddings to detect points where adjacent sentence groups become semantically different.

For SciFact, many abstracts are short, so semantic chunking may often keep the full abstract unchanged. To avoid unstable behavior on very short abstracts, we keep 1–2 sentence abstracts as a single chunk and apply semantic splitting only when an abstract has at least 3 sentences.

This strategy tests whether meaning-based splitting gives better retrieval than full-abstract retrieval, sentence-level retrieval, or token-based chunking.

In [12]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

semantic_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},
)

semantic_splitter = SemanticChunker(
    semantic_embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95,
)

semantic_chunks = []

for doc in tqdm(corpus, desc="Semantic chunking"):
    doc_id = get_doc_id(doc)
    text = make_full_text(doc)
    sentences = get_abstract_sentences(doc)

    if not text:
        continue

    # Semantic splitting is unstable for very short abstracts.
    # For 1-2 sentence abstracts, keep the whole abstract.
    if len(sentences) < 3:
        pieces = [text]
    else:
        try:
            pieces = semantic_splitter.split_text(text)
        except Exception:
            pieces = [text]

    for i, piece in enumerate(pieces):
        semantic_chunks.append({
            "chunk_id": f"{doc_id}_sem_{i}",
            "doc_id": doc_id,
            "chunk_index": i,
            "chunk_type": "semantic",
            "text": piece.strip(),
        })

save_and_report(
    semantic_chunks,
    "Semantic",
    CHUNK_DIR / "chunks_semantic.json"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Semantic chunking: 100%|████████████████████| 5183/5183 [15:20<00:00,  5.63it/s]


=== Semantic ===
  Total chunks:    10504
  Unique docs:     5183 / 5183
  Chunks/doc:      2.03
  Avg words/chunk: 105.9
  Median:          93
  Min / Max:       1 / 1241
  Saved to:        chunks/chunks_semantic.json



# cleaning chunks

In [17]:
import json
import re
from pathlib import Path
from collections import defaultdict

CHUNK_DIR = Path("chunks")

def is_bad_tiny_chunk(text, min_words=4):
    """
    Returns True for tiny/noisy chunks that should not be indexed.
    """
    t = text.strip()
    words = t.split()

    # Empty or extremely tiny chunks
    if len(words) < min_words:
        return True

    # Mostly punctuation
    if re.fullmatch(r"[\W_]+", t):
        return True

    # DOI / URL / availability noise
    lowered = t.lower()
    noisy_patterns = [
        "doi:",
        "http://",
        "https://",
        "availability",
        "copyright",
        "©",
        "table of contents",
        "unique identifier",
    ]

    if any(p in lowered for p in noisy_patterns):
        return True

    # Citation-like fragments, e.g. 2000;283:3211–3216
    if re.fullmatch(r"\d{4};\d+:\d+[–-]\d+\.?", t):
        return True

    return False


def clean_chunks(chunks, min_words=4):
    """
    Removes tiny/noisy chunks, but makes sure every document keeps at least one chunk.
    """
    by_doc = defaultdict(list)

    for c in chunks:
        by_doc[str(c["doc_id"])].append(c)

    cleaned = []

    for doc_id, doc_chunks in by_doc.items():
        kept = [
            c for c in doc_chunks
            if not is_bad_tiny_chunk(c["text"], min_words=min_words)
        ]

        # Safety: never drop an entire document.
        # If all chunks were filtered, keep the longest original chunk.
        if not kept:
            kept = [max(doc_chunks, key=lambda c: len(c["text"].split()))]

        cleaned.extend(kept)

    return cleaned

In [19]:
files_to_clean = [
    "chunks_recursive_128.json",
    "chunks_recursive_256.json",
    "chunks_semantic.json",
]

for fname in files_to_clean:
    path = CHUNK_DIR / fname

    with open(path, "r") as f:
        chunks = json.load(f)

    cleaned = clean_chunks(chunks, min_words=4)

    print(f"{fname}: {len(chunks)} -> {len(cleaned)} chunks removed {len(chunks) - len(cleaned)}")

    with open(path, "w") as f:
        json.dump(cleaned, f, ensure_ascii=False, indent=2)

chunks_recursive_128.json: 18852 -> 18638 chunks removed 214
chunks_recursive_256.json: 10011 -> 9905 chunks removed 106
chunks_semantic.json: 10594 -> 10594 chunks removed 0


# semantic chunk max was 1200+ words. changing that

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

semantic_fallback_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=256,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""],
)

semantic_path = CHUNK_DIR / "chunks_semantic.json"

with open(semantic_path, "r") as f:
    semantic_chunks = json.load(f)

fixed_semantic_chunks = []

for c in semantic_chunks:
    text = c["text"].strip()

    # If semantic chunk is too long, split it recursively.
    if len(text.split()) > 300:
        pieces = semantic_fallback_splitter.split_text(text)

        for i, piece in enumerate(pieces):
            new_c = c.copy()
            new_c["chunk_id"] = f"{c['chunk_id']}_fb_{i}"
            new_c["chunk_index"] = i
            new_c["chunk_type"] = "semantic_with_recursive_fallback"
            new_c["text"] = piece.strip()
            fixed_semantic_chunks.append(new_c)
    else:
        fixed_semantic_chunks.append(c)

fixed_semantic_chunks = clean_chunks(fixed_semantic_chunks, min_words=4)

with open(semantic_path, "w") as f:
    json.dump(fixed_semantic_chunks, f, ensure_ascii=False, indent=2)

print(f"Semantic chunks fixed: {len(semantic_chunks)} -> {len(fixed_semantic_chunks)}")

Semantic chunks fixed: 10504 -> 10594


## 7. Side-by-side comparison

Quick at-a-glance summary so you can see how chunk counts and granularity differ across strategies. The strategy with the right balance — enough granularity to be precise, not so much that signal gets fragmented — is what we're hunting for on Day 4.

In [20]:
chunk_files = {
    "Document-level": CHUNK_DIR / "chunks_document.json",
    "Sentence-level": CHUNK_DIR / "chunks_sentence.json",
    "Fixed 128": CHUNK_DIR / "chunks_fixed_128.json",
    "Recursive 128": CHUNK_DIR / "chunks_recursive_128.json",
    "Recursive 256": CHUNK_DIR / "chunks_recursive_256.json",
    "Semantic": CHUNK_DIR / "chunks_semantic.json",
}

summary = []

for name, path in chunk_files.items():
    with open(path, "r") as f:
        chunks = json.load(f)

    lengths = np.array([len(c["text"].split()) for c in chunks])
    docs_covered = len({str(c["doc_id"]) for c in chunks})

    summary.append({
        "Strategy": name,
        "Total chunks": len(chunks),
        "Unique docs": docs_covered,
        "Chunks/doc": round(len(chunks) / len(corpus), 2),
        "Avg words": round(float(lengths.mean()), 1),
        "Median words": int(np.median(lengths)),
        "Min words": int(lengths.min()),
        "Max words": int(lengths.max()),
    })

summary_df = pd.DataFrame(summary)
summary_df

,Strategy,Total chunks,Unique docs,Chunks/doc,Avg words,Median words,Min words,Max words
0,Document-level,5183,5183,1.00,214.6,204,33,1541
1,Sentence-level,45952,5183,8.87,35.9,35,3,221
2,Fixed 128,17400,5183,3.36,73.3,78,5,125
3,Recursive 128,18638,5183,3.60,60.8,63,4,117
4,Recursive 256,9905,5183,1.91,115.5,125,5,223
5,Semantic,10594,5183,2.04,103.8,95,4,300


In [14]:
summary_df.to_csv(CHUNK_DIR / "chunking_summary.csv", index=False)
print(f"Saved summary to {CHUNK_DIR / 'chunking_summary.csv'}")

Saved summary to chunks/chunking_summary.csv


# diagnostic cell to see the chunk content

In [15]:
import json
from pathlib import Path

CHUNK_DIR = Path("chunks")

for fname in [
    "chunks_recursive_128.json",
    "chunks_recursive_256.json",
    "chunks_semantic.json",
]:
    path = CHUNK_DIR / fname
    with open(path) as f:
        chunks = json.load(f)

    tiny = [c for c in chunks if len(c["text"].split()) <= 3]

    print(f"\n=== {fname} ===")
    print(f"Tiny chunks <= 3 words: {len(tiny)}")

    for c in tiny[:10]:
        print(c["chunk_id"], "|", repr(c["text"]))


=== chunks_recursive_128.json ===
Tiny chunks <= 3 words: 65
54440_rec128_3 | '.'
665817_rec128_1 | '.'
1456068_rec128_1 | '.'
1780819_rec128_1 | '.'
1991105_rec128_2 | '. DOI:http://dx.doi.org/10.7554/eLife.00422.001.'
2107238_rec128_1 | '.    AVAILABILITY http://samtools.sourceforge.net.'
3210545_rec128_3 | '.'
3752408_rec128_1 | '.'
4489217_rec128_4 | '.'
5114940_rec128_5 | '.'

=== chunks_recursive_256.json ===
Tiny chunks <= 3 words: 11
79447_rec256_2 | '.'
2488880_rec256_2 | '.'
9167230_rec256_2 | 'FUNDING WHO.'
9764256_rec256_6 | '.'
9822397_rec256_2 | '.'
13083189_rec256_2 | '.'
15997009_rec256_2 | '.'
16322674_rec256_3 | '.'
34268160_rec256_2 | '.'
37336085_rec256_1 | '.'

=== chunks_semantic.json ===
Tiny chunks <= 3 words: 49
2225918_sem_1 | 'PAPERCLIP.'
4465735_sem_1 | 'difficile infection.'
4687948_sem_1 | '2000;283:3211-3216'
6540064_sem_1 | 'Unique identifier: NCT01959971.'
7074001_sem_0 | 'Evidence into practice.'
9460704_sem_1 | '©2013 AACR.'
10300888_sem_1 | 'PAPERCL

## Next: Day 3 — Indexing

With four chunked corpora saved, Day 3 is to embed each one with `bge-small-en-v1.5` and build four separate vector databases (FAISS or Chroma both work locally). Then Day 4 runs the 338 validation claims through each index and computes Hit Rate and MRR.